# DiD Modelling

This notebook explains, in simple terms, how the data is prepared for a Difference-in-Differences (DiD) analysis of the Job-ready Graduates Program (JRGP).

## 1) What we are trying to estimate

Goal: estimate how much Australia's enrolments changed after the Job-Ready Graduates Package (JRG, from 2021 onward), relative to two control countries — the **UK** and **New Zealand** — neither of which introduced an equivalent structural policy over the same period.

In plain language: "Did Australia move differently from the UK and NZ after the policy started, and by how much?"

## 2) How the data was modelled for DiD

The data is prepared into a balanced panel with one row per **country × year** for each discipline.

Main setup steps:

1. **Harmonise fields:** map AUS, UK (HESA), and NZ (TEC) disciplines to common category keys (1–11).
2. **Align time:** convert UK academic years (start-year convention: `2016/17 → 2016`) and NZ calendar years to a common annual format.
3. **Keep comparable period:** use the overlapping window 2016–2024 where UK data is available from 2016; otherwise 2019–2024. AUS and NZ both cover 2016–2024 for all disciplines.
4. **Aggregate to common unit:** country-year enrolment totals per discipline (NZ uses `total_bachelors`; some UK categories require groupby-sum across multiple HESA subject rows).
5. **Build treatment indicators:**
   - `treated = 1` for AUS, `0` for UK and NZ (both are untreated controls)
   - `nz_dummy = 1` for NZ, `0` otherwise (UK is the reference country for level differences)
   - `post = 1` for 2021+ (JRG period), `0` otherwise
   - `did = treated × post` (the main DiD interaction)
6. **Add AUS-specific COVID controls** (`treated_covid2020`, `treated_covid2021`) to separate the AUS pandemic shock from the policy effect.
7. **Use log outcome:** `y = log(enrolments)`, so coefficients are interpretable as approximate percentage changes.

## 3) Why these modelling choices are appropriate

- **DiD vs levels:** DiD compares *changes* over time, so permanent country-level differences (AUS being larger than NZ, UK having different base rates) do not bias the estimate. Country fixed effects absorb any stable cross-country gap.
- **Year fixed effects:** absorbed by `C(year)` in OLS, these account for global shocks common to all three countries in the same year — including COVID-related disruptions visible across AUS, UK, and NZ simultaneously.
- **AUS-specific COVID controls:** `treated_covid2020` and `treated_covid2021` separate the AUS-specific pandemic impact (lockdowns, border closures) from the JRG policy effect, which starts in the same period.
- **Two control countries:** using both UK and NZ strengthens identification. If AUS diverges from *both* UK and NZ after 2021, the JRG mechanism is more credibly isolated than with a single comparator. NZ also serves as a Pacific-proximate control with demographic similarities to AUS.
- **HC3 robust SEs:** heteroscedasticity-robust (HC3) standard errors are used throughout, as required for small panels where cross-country variance in enrolment levels may differ substantially.
- **Log-linear specification:** preferred for count outcomes to ensure proportional interpretation and reduce the influence of scale differences between AUS, UK, and NZ enrolment totals.
- **Event study checks:** year-by-year DiD point estimates relative to the 2020 baseline assess whether AUS tracked the pooled UK + NZ trend during the pre-treatment period (2016–2019).

## 4) Regression model (per-discipline, two-way fixed effects)

Each discipline is estimated separately. The primary estimating equation is:

$$\log(E_{ct}) = \beta_0 + \beta_1 \cdot \text{Treated}_c + \beta_2 \cdot \text{NZ}_c + \beta_3 \cdot (\text{Treated}_c \times \text{Post}_t) + \sum_{t} \gamma_t \cdot \mathbf{1}_{[\text{year}=t]} + \varepsilon_{ct}$$

where $c \in \{\text{AUS, UK, NZ}\}$, $t$ indexes years, and:

| Term | Variable | Definition |
|------|----------|------------|
| $\beta_0$ | Intercept | UK baseline log-enrolments in the reference year |
| $\beta_1 \cdot \text{Treated}_c$ | AUS country FE | $= 1$ if AUS, $0$ otherwise |
| $\beta_2 \cdot \text{NZ}_c$ | NZ country FE | $= 1$ if NZ, $0$ otherwise (UK = reference) |
| $\text{Post}_t$ | — | $= 1$ if $t \geq 2021$, else $0$ |
| $\beta_3 \cdot (\text{Treated}_c \times \text{Post}_t)$ | **DiD term** | Main JRG effect — coefficient of interest |
| $\gamma_t$ | Year FEs | Absorbed common time shocks across all three countries |
| $\varepsilon_{ct}$ | Residual | HC3 heteroscedasticity-robust standard errors |

Implemented in statsmodels as:

```python
formula = "log_enrollments ~ treated + nz_dummy + did + C(year)"
model   = smf.ols(formula, data=panel).fit(cov_type="HC3")
```

where `did = treated * post`. Cross-checked against `linearmodels.PanelOLS` with entity and time effects — estimates match.

## 5) Interpreting the coefficient

$$\hat{\beta}_3 \approx \frac{\Delta \log E_{\text{AUS}}}{\Delta \text{Post}} - \frac{\Delta \log E_{\text{UK+NZ}}}{\Delta \text{Post}}$$

- **$\hat{\beta}_3 < 0$:** AUS enrolments grew more slowly (or declined) relative to the pooled UK + NZ trend after JRG — a negative policy effect on this discipline.
- **$\hat{\beta}_3 > 0$:** AUS enrolments grew faster than the pooled UK + NZ trend after JRG — a positive policy effect.
- **Approximate % effect:** $(e^{\hat{\beta}_3} - 1) \times 100$

Statistical significance is assessed against the null $H_0: \beta_3 = 0$ using HC3 robust standard errors with degrees of freedom $= N - k$ (where $N$ is panel observations and $k$ is the number of estimated parameters).

## 6) Assumption to state in write-up

**Parallel trends assumption:** without JRG, AUS enrolments in each discipline would have followed a similar trend to the pooled UK and NZ trend after 2021.

This is tested with:
- **Pre-trend event study plots** — year-by-year DiD coefficients relative to the 2020 baseline across 2016–2019 should be close to zero if the assumption holds.
- **Placebo test** — a fake structural break placed at 2019 in the AUS pre-period (2016–2020 only) should be statistically insignificant.

The assumption also requires that neither UK nor NZ introduced a comparable structural higher-education policy change over the 2021–2024 period. Based on available evidence, no equivalent fee or funding restructuring occurred in the UK or NZ during this window.